In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import pitch_name_to_pitch


def resample(x: pq.Audio, ratio: float) -> pq.Audio:
    """Read x at `ratio` times its original speed, with linear interpolation."""
    samples = np.asarray(x.samples)
    N = len(samples)
    M = int(N / ratio)                        # the duration scales by 1 / ratio
    p = np.arange(M) * ratio                  # fractional read positions in x
    p_floor = np.floor(p).astype(int)
    alpha = (p - p_floor)[:, np.newaxis]
    p_next = np.minimum(p_floor + 1, N - 1)
    y = (1 - alpha) * samples[p_floor] + alpha * samples[p_next]
    return pq.Audio(y, x.sample_rate)


def repitch(x: pq.Audio, k: float) -> pq.Audio:
    """Shift the pitch of x by k steps."""
    return resample(x, 2 ** (k / 12))

In [ ]:
LOOP_START, LOOP_END = 0.499, 0.720   # seconds: 206 periods of A#5, about 9744 audio samples
RELEASE = 3.704                       # seconds: where the release begins


def clarinet(pitch: str, duration: float) -> pq.Audio:
    """A sampled clarinet with independent control over pitch and duration."""
    recording = pq.Audio.from_file("../assets/clarinet.wav")
    k = pitch_name_to_pitch(pitch) - pitch_name_to_pitch("A#5")
    x = np.asarray(repitch(recording, k).samples)
    # Reading the recording 2^(k/12) times faster also moves the loop points
    to_samples = recording.sample_rate / 2 ** (k / 12)
    s, e, r = (round(t * to_samples) for t in (LOOP_START, LOOP_END, RELEASE))
    # Play up to the loop, repeat the loop to fill out the duration, then play the release
    n = max(0, round((duration * recording.sample_rate - s - (len(x) - r)) / (e - s)))
    return pq.Audio(np.concatenate([x[:s]] + [x[s:e]] * n + [x[r:]]), recording.sample_rate)


# The same pitch at three durations, then three pitches at the same duration
notes = [("A#5", 2.0), ("A#5", 4.0), ("A#5", 8.0), ("D5", 3.0), ("F5", 3.0), ("D6", 3.0)]
onsets = np.cumsum([0.0] + [duration for _, duration in notes[:-1]])
melody = pq.Score([(t, {"pitch": pitch, "duration": duration})
                   for t, (pitch, duration) in zip(onsets, notes)])
pq.play(melody.render(clarinet), normalize=True)